<a href="https://colab.research.google.com/github/prabhnoor14/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(total_revenue, total_units)
print(f"Across all 400 orders, vendors generated ${total_revenue:,.2f} in revenue from {total_units} units sold.")

8520.0 783
Across all 400 orders, vendors generated $8,520.00 in revenue from 783 units sold.


todo explaination: revenue = qty * price is a row-wise multiplication, so it computes a revenue value for every one of the 400 orders. This creates a new column without adding or removing any rows. The total came out to 8,520.00, which falls within the expected 8,000-9,000 range, so this looks correct.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO
by_category = df.groupby('category')['revenue'].sum()
by_category = by_category.sort_values(ascending=False)
by_category = by_category.to_frame()

by_category['share'] = (by_category['revenue'] / total_revenue) * 100

by_category


,revenue,share
category,,
Food,4293.0,50.387324
Merch,1771.5,20.792254
Drink,1554.0,18.239437
RainGear,901.5,10.580986


todo explaination: I grouped the orders by category and summed up revenue in each group, then sorted that from highest to lowest so the report reads in order. I used to_frame() because grouping and summing a single column gives back a Series, not a DataFrame, and I needed it as a DataFrame so I could add a share column next to revenue. Then I calculated each category's share of total revenue as a percentage. Food has 50.4% of total revenue,meaning the vendors' business is food-driven. This also aligns with how df was created and p for food was .5

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO
avg = df.groupby('vendor_id')['revenue'].mean()
avg = avg.rename('avg_revenue')
avg = avg.sort_values(ascending=False)
avg = avg.to_frame()

avg['count'] = df.groupby('vendor_id')['revenue'].count()
top_vendor = avg.index[0]
top_avg_revenue = avg.iloc[0]['avg_revenue']

avg


,avg_revenue,count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


In [5]:
print(f"{top_vendor} had the highest average order revenue at ${top_avg_revenue:.2f}.")

V-01 had the highest average order revenue at $22.60.


todo explaination: I grouped the orders by vendor and found the average revenue per order for each one, then sorted that from highest to lowest. I also counted how many orders each vendor had. V-01 has the highest average order revenue, but with only 94 orders compared to some vendors having over 100.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [6]:
# TODO
merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()
merch_share = (merch_revenue / total_revenue) * 100
print(f"Merch revenue accounts for {merch_share:.1f}% of total revenue.")

Merch revenue accounts for 20.8% of total revenue.


 todo explaination: I filtered the df to just Merch rows, summed the revenue, then divided by total revenue to get the share. The result was 20.8%, which matches the Merch row from Q2.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [7]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

rows_match = len(joined) == len(df)
revenue_match = joined['revenue'].sum() == df['revenue'].sum()

print(f'rows match: {rows_match}')
print(f'revenue match: {revenue_match}')

unmatched = joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique()
print(f'unmatched vendor: {unmatched}')

rows match: True
revenue match: True
unmatched vendor: ['V-18']


**The unmatched vendor is v-18, and what I did about it:  since V-18 isn't in the vendor_names lookup, the merge left its vendor_name blank (NaN) for every order from that vendor. I filled those blanks with the vendor_id itself instead of leaving them empty, so the report still shows something readable

In [8]:
joined['vendor_name'] = joined['vendor_name'].fillna(joined['vendor_id'])
print(joined['vendor_name'].isna().sum())

0


 todo explaination: I merged in the vendor_names lookup with a left join so every order from df stays even if there's no matching name. validate='many_to_one' checks that vendor_names doesn't have duplicate ids, which would've messed up the row count. I confirmed the merge didn't change anything by checking that the row count and revenue total both still match df. Then I found V-18 was the vendor with no match, since its vendor_name came back blank, and I filled those blanks with the vendor_id itself so the report doesn't have empty cells, which is what the 0 being printed in the previous cell shows.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [9]:
# TODO
pivot = pd.pivot_table(joined, values='revenue', index='vendor_name', columns='category', aggfunc='sum', margins=True)
pivot

category,Drink,Food,Merch,RainGear,All
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
V-18,582.0,1018.5,508.5,240.0,2349.0
All,1554.0,4293.0,1771.5,901.5,8520.0


todo explaination: I made a pivot table using joined so it shows vendor names instead of ids, with vendor_name as rows and category as columns, and revenue summed in each cell. margins=True adds row and column totals automatically. The grand total in the bottom right is 8,520, which matches the Q1 total

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [10]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) Rotunda Tacos had the lowest total revenue (1,914). Rotunda Taco's food revenue was only 882. That is the lowest of any vendor. Food is the biggest category overall,making up 50.4% of revenue. Rotunda Tacos should sell more food since that's where most of the money is being spent.

b) Q5 is my least trustworthy answer. V-18 was not in the vendor_names lookup,meaning I dont know what business it is. I subbed with "V-18" instead of a real name to avoid blank cells. Someone reading the report might not even notice it's not a real vendor name, which might lead to misinterpretation of the report.